# Gold Metadata layer v7 — generic title/author split, strict language, local Ollama

This metadata layer is strict and generic for Dutch and English documents.

Changes in v7:
- No document-specific terms or rules.
- No subtitle field.
- Organizations are not used as final KMP metadata by default.
- Stronger generic title/front-matter parsing: removes dates, IDs, organizations and trailing author names from titles.
- Forces summaries and descriptions to follow the detected document language.
- Keeps 10–15 keywords when enough evidence exists.
- Compatible with larger Ollama models later by changing `LOCAL_MODEL`.


Changes in v8:
- Generic trailing-author split now tries the shortest plausible person name first.
- Prevents title words from being swallowed into the author field.
- Prefers clean front-matter titles when model output ends with a preposition or is shorter/truncated.


In [52]:

from pathlib import Path
import json
import re
import time
import requests
from datetime import datetime

# =========================
# Configuration
# =========================
DOCUMENT_ID = "doc_01"

LOCAL_MODEL = "qwen2.5:3b-instruct"   # Later: "qwen2.5:7b-instruct" or "qwen2.5:14b-instruct"
OLLAMA_URL = "http://localhost:11434/api/generate"

BASE_DIR = Path("../../Data")
SILVER_DIR = BASE_DIR / "silver"
SILVER_NLP_DIR = BASE_DIR / "silver_nlp"
GOLD_DIR = BASE_DIR / "gold"
META_DIR = BASE_DIR / "gold_meta"
META_DIR.mkdir(parents=True, exist_ok=True)

REQUEST_TIMEOUT_SECONDS = 900
TEMPERATURE = 0.0
NUM_CTX = 8192

def read_json(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def write_json(obj, path: Path):
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=4)

def find_file(directory: Path, patterns):
    for p in patterns:
        matches = sorted(directory.glob(p))
        if matches:
            return matches[0]
    raise FileNotFoundError(f"No file found in {directory} for patterns: {patterns}")

silver_path = find_file(SILVER_DIR, [f"{DOCUMENT_ID}_silver.json", "*_silver.json"])
silver_nlp_path = find_file(SILVER_NLP_DIR, [f"{DOCUMENT_ID}_silver_nlp.json", "*_silver_nlp.json"])
gold_path = find_file(GOLD_DIR, [f"{DOCUMENT_ID}_gold.json", "*_gold.json"])

silver = read_json(silver_path)
silver_nlp = read_json(silver_nlp_path)
gold = read_json(gold_path)

detected_language = silver.get("detected_language") or silver_nlp.get("detected_language") or gold.get("detected_language") or "unknown"

print("Loaded:")
print("Silver:", silver_path)
print("Silver NLP:", silver_nlp_path)
print("Gold:", gold_path)
print("Language:", detected_language)


Loaded:
Silver: ..\..\Data\silver\doc_01_silver.json
Silver NLP: ..\..\Data\silver_nlp\doc_01_silver_nlp.json
Gold: ..\..\Data\gold\doc_01_gold.json
Language: en


In [53]:

# =========================
# Generic evidence helpers
# =========================

def clean_line(line):
    line = re.sub(r"\s+", " ", str(line or "")).strip()
    return line

def truncate(text, n):
    text = re.sub(r"\s+", " ", str(text or "")).strip()
    if len(text) <= n:
        return text
    return text[:n].rsplit(" ", 1)[0] + "..."

def get_titlepage_excerpt(silver_obj, max_chars=3500):
    # Prefer explicit titlepage/front matter fields if present.
    for key in ["titlepage_text", "front_matter", "first_page_text"]:
        val = silver_obj.get(key)
        if isinstance(val, str) and val.strip():
            return val[:max_chars]

    parts = silver_obj.get("document_parts", {})
    if isinstance(parts, dict):
        for key in ["titlepage_text", "front_matter", "raw_start_text"]:
            val = parts.get(key)
            if isinstance(val, str) and val.strip():
                return val[:max_chars]
        main_text = parts.get("main_text", "")
        if main_text:
            return main_text[:max_chars]

    main_text = silver_obj.get("main_text", "")
    if main_text:
        return main_text[:max_chars]

    return ""

def extract_regex_candidates(text):
    lines = [clean_line(x) for x in str(text or "").splitlines()]
    lines = [x for x in lines if x]

    # Generic date patterns for NL/EN documents.
    date_patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        r"\b\d{1,2}\s+(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+\d{4}\b",
        r"\b\d{1,2}\s+(?:january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}\b",
        r"\b(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december)\s+\d{4}\b",
        r"\b(?:january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}\b",
        r"\b(?:19|20)\d{2}\b"
    ]

    dates = []
    for pat in date_patterns:
        for m in re.finditer(pat, text, flags=re.I):
            val = m.group(0).strip()
            if val not in dates:
                dates.append(val)

    emails = sorted(set(re.findall(r"[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}", text)))
    urls = sorted(set(re.findall(r"https?://\S+|www\.\S+", text)))

    org_markers = [
        "university","universiteit","hogeschool","college","school","institute","instituut",
        "province","provincie","municipality","gemeente","ministry","ministerie","department",
        "bedrijf","company","bv","b.v.","nv","n.v.","ltd","inc","gmbh","foundation","stichting",
        "research","lectoraat","consortium","agency","centre","center"
    ]

    organization_candidates = []
    for line in lines[:40]:
        low = line.lower()
        if any(marker in low for marker in org_markers) and len(line) <= 140:
            if line not in organization_candidates:
                organization_candidates.append(line)

    # Author candidates: generic line heuristic, not document-specific.
    # Candidate has 2-5 name-like words, no org marker, no date, not too long.
    author_candidates = []
    for line in lines[:30]:
        low = line.lower()
        if any(marker in low for marker in org_markers):
            continue
        if any(d in line for d in dates):
            continue
        if len(line) > 80:
            continue
        words = line.split()
        if 2 <= len(words) <= 5:
            name_like = sum(1 for w in words if re.match(r"^[A-ZÀ-Ö][A-Za-zÀ-ÖØ-öø-ÿ'\-]+$", w))
            if name_like >= 2 and line not in author_candidates:
                author_candidates.append(line)

    # Title candidates: early non-date, non-org lines, usually before first obvious author/date.
    title_candidates = []
    for line in lines[:12]:
        low = line.lower()
        if any(marker in low for marker in org_markers):
            continue
        if any(d in line for d in dates):
            continue
        if "@" in line or re.search(r"\d{1,2}[/-]\d{1,2}[/-]\d{2,4}", line):
            continue
        if 4 <= len(line) <= 140:
            if line not in title_candidates:
                title_candidates.append(line)

    return {
        "title_candidates": title_candidates[:5],
        "author_candidates": author_candidates[:5],
        "organization_candidates": organization_candidates[:8],
        "date_candidates": dates[:8],
        "emails": emails,
        "urls": urls
    }

titlepage_excerpt = get_titlepage_excerpt(silver)
regex_candidates = extract_regex_candidates(titlepage_excerpt)

silver_candidates = silver.get("titlepage_candidates") or silver_nlp.get("titlepage_candidates_from_silver") or {}
gold_summary = gold.get("document_summary", "")
gold_top_terms = gold.get("top_terms", [])

print("Regex candidates:")
print(json.dumps(regex_candidates, ensure_ascii=False, indent=2))


Regex candidates:
{
  "title_candidates": [
    "A Deep Neural Network for",
    "the Detection of Epileptic Seizures",
    "in Comatose Pediatric Patients",
    "BSc Graduation Thesis",
    "Angélique Huige (80639)"
  ],
  "author_candidates": [
    "A Deep Neural Network for",
    "the Detection of Epileptic Seizures",
    "in Comatose Pediatric Patients",
    "BSc Graduation Thesis",
    "Angélique Huige (80639)"
  ],
  "organization_candidates": [
    "Department of Neurology",
    "HZ University of Applied Sciences",
    "researched the possibilities of applying data science and deep learning for the analysis of EEG data.",
    "University of Applied Sciences, for which I conducted an internship at Erasmus MC from September",
    "This research allowed me to pursue my interests in cognitive sciences and gain invaluable experience in",
    "instrumental in shaping this research and my learning journey. Robert gave me, without exception,",
    "incredibly insightful feedback and com

In [54]:

# =========================
# Build compact metadata evidence
# =========================

evidence = {
    "detected_language": detected_language,
    "titlepage_excerpt": titlepage_excerpt[:3500],
    "regex_candidates_from_titlepage": regex_candidates,
    "silver_titlepage_candidates": silver_candidates,
    "gold_summary": truncate(gold_summary, 1500),
    "gold_top_terms": gold_top_terms[:15],
    "gold_main_topics": gold.get("main_topics", [])[:10],
    "gold_results_or_conclusions": gold.get("results_or_conclusions", [])[:10],
}

evidence_text = json.dumps(evidence, ensure_ascii=False, indent=2)
print("Evidence chars:", len(evidence_text))
print(evidence_text[:1800])


Evidence chars: 12093
{
  "detected_language": "en",
  "titlepage_excerpt": "A Deep Neural Network for\nthe Detection of Epileptic Seizures\nin Comatose Pediatric Patients\nBSc Graduation Thesis\nAngélique Huige (80639)\nJanuary 14th, 2024\nSubmitted in partial fulfillment of the requirements\nfor the degree of BSc Data Science\nSupervised by:\nRobert van den Berg, MD, PhD (Erasmus MC)\nBente Sinke, MSc (HZ UAS)\nJolène Cijsouw, MSc (HZ UAS)\nErasmus MC\nDepartment of Neurology\nHZ University of Applied Sciences\nInformation and Communication Technology\n\nPreface\nEducation never ends, Watson. It is a\nseries of lessons, with the greatest for the\nlast.\nSherlock Holmes\nAs I stand at the end of an educational chapter, it is with both gratitude and pleasure that I present my\nbachelor thesis: \"A Deep Neural Network for the Detection of Epileptic Seizures in Comatose Pediatric\nPatients\". This work represents several months of exploration, dedication, curiosity, and passion as I\nres

In [55]:

# =========================
# Ollama call
# =========================

schema = {
    "title": None,
    "authors": [],
    "date": None,
    "document_type": None,
    "language": None,
    "research_or_project_topic": None,
    "research_question_or_goal": None,
    "short_summary": None,
    "keywords": [],
    "tools_technologies_or_models": [],
    "main_outputs_or_results": [],
    "suitable_kmp_fields": {
        "title": None,
        "description": None,
        "keywords": [],
        "contributors": [],
        "date": None,
        "language": None,
        "document_type": None
    },
    "contact": {
        "name": None,
        "email": None,
        "phone": None
    },
    "confidence_notes": []
}

def check_ollama():
    try:
        r = requests.get("http://localhost:11434/api/tags", timeout=5)
        r.raise_for_status()
        return True
    except Exception as e:
        print("Ollama is not reachable. Start it with: ollama serve")
        print("Error:", e)
        return False

def ollama_generate_json(prompt):
    payload = {
        "model": LOCAL_MODEL,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        "options": {
            "temperature": TEMPERATURE,
            "num_ctx": NUM_CTX
        }
    }
    r = requests.post(OLLAMA_URL, json=payload, timeout=REQUEST_TIMEOUT_SECONDS)
    r.raise_for_status()
    return r.json().get("response", "")

def parse_json_safely(text):
    text = str(text or "").strip()
    try:
        return json.loads(text)
    except Exception:
        m = re.search(r"\{.*\}", text, flags=re.S)
        if m:
            return json.loads(m.group(0))
    raise ValueError("Could not parse JSON")

output_language_instruction = (
    "Write short_summary, research_or_project_topic, research_question_or_goal and confidence_notes in Dutch. Keep technical terms in their original form when appropriate."
    if str(detected_language).lower().startswith("nl")
    else "Write short_summary, research_or_project_topic, research_question_or_goal and confidence_notes in English."
)

prompt = f"""
You are a strict metadata extraction system for a knowledge management platform.

Return valid JSON only.

Mandatory language rule:
- Detected language: {detected_language}.
- {output_language_instruction}
- Do not write an English summary for a Dutch document. Use natural, fluent language, not literal translations.

Rules:
- Extract metadata only when there is evidence in the provided text.
- Do not invent titles, names, dates, phone numbers, emails, or fields.
- Do not output a subtitle field.
- Organizations are not required for final metadata; leave them out of suitable_kmp_fields.
- If a field is not explicitly supported by evidence, use null or [].
- The title usually comes from the title page/front matter, not from later headings. Do not include dates, student numbers, author names, page numbers, organizations, or locations in the title.
- People and dates from Silver NLP are suggestions only; verify against evidence.
- Return 10 to 15 keywords when enough evidence exists.
- Keywords must be meaningful terms, not generic function words.
- Do not summarize appendices as separate metadata fields.
- Return exactly one JSON object matching this schema:
{json.dumps(schema, ensure_ascii=False, indent=2)}

EVIDENCE:
{evidence_text}
"""

if not check_ollama():
    raise RuntimeError("Ollama is not running.")

start = time.time()
raw_response = ollama_generate_json(prompt)
runtime = round(time.time() - start, 2)
print("Runtime seconds:", runtime)
print(raw_response[:1200])



Runtime seconds: 7.18
{
  "short_summary": "This document discusses the development of a deep learning binary classification model aimed at detecting epileptic seizures in pediatric comatose patients. The research explores various approaches, including convolutional neural networks and recurrent neural networks, to analyze minimally processed EEG data.",
  "research_or_project_topic": "Development of Deep Learning Models for Epileptic Seizure Detection in Comatose Pediatric Patients",
  "research_question_or_goal": "How can deep learning be applied to minimally processed EEG data for the detection of epileptic seizures in pediatric comatose patients?",
  "keywords": [
    "deep learning",
    "convolutional neural networks (CNN)",
    "recurrent neural networks (RNN)",
    "epileptic seizures",
    "minimally processed data",
    "EEG analysis",
    "comatose patients",
    "binary classification model",
    "seizure detection",
    "pediatric neurology"
  ]
}


In [56]:
# =========================
# Strict validation / anti-hallucination
# =========================

metadata = parse_json_safely(raw_response)

GENERIC_KEYWORD_STOPWORDS = STOPWORDS if "STOPWORDS" in globals() else {
    "de","het","een","en","of","in","op","te","van","voor","met","dat","dit","die","als","aan","om","er",
    "is","zijn","wordt","worden","was","waren","heeft","hebben","door","bij","uit","naar","ook","maar",
    "wat","hoe","waar","welke","moet","moeten","tijdens","alleen","the","a","an","and","or","in","on","to",
    "of","for","with","that","this","as","by","from","what","how","which","during","only"
}
BAD_KEYWORD_MARKERS = {"figure_caption", "table_start", "table_end", "bijlage", "appendix", "references", "bibliografie"}
ORG_MARKERS = {
    "university","universiteit","hogeschool","college","school","institute","instituut",
    "province","provincie","municipality","gemeente","ministry","ministerie","department",
    "bedrijf","company","bv","b.v.","nv","n.v.","ltd","inc","gmbh","foundation","stichting",
    "research","lectoraat","consortium","agency","centre","center","applied sciences"
}
TITLE_STOP_SECTIONS = {"voorwoord","preface","abstract","samenvatting","inhoudsopgave","contents","table of contents"}
ACRONYMS_TO_KEEP = {"AI","ML","NLP","LLM","CPU","GPU","API","UI","UX","JSON","HTTP","HTTPS","FTP","PDF","PET","WMS","WFS","GIS","ISO","SQL","HTML","CSS","JS"}
SMALL_TITLE_WORDS = {"de","het","een","en","of","van","voor","met","in","op","aan","om","the","a","an","and","or","of","for","with","in","on","to","at","by"}


def ensure_list(x):
    if x is None:
        return []
    if isinstance(x, list):
        return [v for v in x if v not in [None, ""]]
    if isinstance(x, str) and x.strip():
        return [x.strip()]
    return []


def evidence_contains(value):
    if not value:
        return False
    return str(value).lower() in evidence_text.lower()


def clean_scalar(value):
    if value is None:
        return None
    value = re.sub(r"\s+", " ", str(value)).strip()
    if value in ["", "-", "null", "None"]:
        return None
    return value


def is_org_line(line):
    low = str(line or "").lower()
    return any(marker in low for marker in ORG_MARKERS)


def is_identifier_line(line):
    line = clean_scalar(line) or ""
    if not line:
        return True
    # Generic IDs: mostly numeric, student numbers, page numbers, registration numbers.
    if re.fullmatch(r"[#A-Za-z]{0,4}\s*\d{4,12}", line):
        return True
    if re.fullmatch(r"\d{4,12}", line):
        return True
    if re.fullmatch(r"\d+\s*[|/\\]\s*\w+", line):
        return True
    return False


def extract_date_from_line(line):
    line = str(line or "")
    patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        r"\b\d{4}[/-]\d{1,2}[/-]\d{1,2}\b",
        r"\b\d{1,2}\s+(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december|january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}\b",
        r"\b(?:januari|februari|maart|april|mei|juni|juli|augustus|september|oktober|november|december|january|february|march|april|may|june|july|august|september|october|november|december)\s+\d{4}\b",
    ]
    for pat in patterns:
        m = re.search(pat, line, flags=re.I)
        if m:
            return m.group(0)
    return None


def smart_title_case(text):
    text = clean_scalar(text)
    if not text:
        return None
    # Only alter lines that are mostly uppercase or have many uppercase words.
    letters = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ]", text)
    mostly_upper = bool(letters) and sum(1 for c in letters if c.isupper()) / len(letters) > 0.65
    if not mostly_upper:
        # Still normalize excessive mixed all-caps tokens like VAN/VOOR without touching normal title casing.
        words = []
        for i, w in enumerate(text.split()):
            raw = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ]", "", w)
            if raw.upper() in ACRONYMS_TO_KEEP:
                words.append(w.upper())
            elif raw.isupper() and len(raw) > 1:
                lw = w.lower()
                words.append(lw if lw in SMALL_TITLE_WORDS and i > 0 else lw.capitalize())
            else:
                words.append(w)
        return " ".join(words)

    out = []
    for i, w in enumerate(text.split()):
        prefix = re.match(r"^\W*", w).group(0)
        suffix = re.search(r"\W*$", w).group(0)
        core = w[len(prefix): len(w)-len(suffix) if suffix else len(w)]
        if not core:
            out.append(w)
            continue
        upper_core = core.upper()
        low_core = core.lower()
        if upper_core in ACRONYMS_TO_KEEP:
            new_core = upper_core
        elif low_core in SMALL_TITLE_WORDS and i > 0:
            new_core = low_core
        else:
            new_core = low_core[:1].upper() + low_core[1:]
        out.append(prefix + new_core + suffix)
    return " ".join(out)


def looks_like_person_name(line):
    line = clean_scalar(line)
    if not line or is_identifier_line(line) or is_org_line(line) or extract_date_from_line(line):
        return False
    words = line.split()
    if not 2 <= len(words) <= 5:
        return False
    # Avoid lines that look like titles with generic title words or punctuation.
    if any(w.lower() in SMALL_TITLE_WORDS for w in words):
        return False
    if any(ch in line for ch in [":", ";", "?", "!"]):
        return False
    name_like = 0
    for w in words:
        core = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ'\-]", "", w)
        if not core:
            continue
        if core.isupper() or core[:1].isupper():
            name_like += 1
    return name_like >= max(2, len(words) - 1)


def clean_title_value(title, authors=None, dates=None):
    title = clean_scalar(title)
    if not title:
        return None
    authors = authors or []
    dates = dates or []
    # Remove emails/URLs and internal markers.
    title = re.sub(r"\S+@\S+|https?://\S+|www\.\S+", "", title)
    title = re.sub(r"\b(?:figure_caption|table_start|table_end)\b", "", title, flags=re.I)
    # Remove dates and standalone years/IDs at boundaries.
    for d in dates:
        if d:
            title = title.replace(d, " ")
    title = re.sub(r"^\s*\d{4}\s+", "", title)
    title = re.sub(r"\s+\d{4,12}\s*$", "", title)
    # Remove author names if the title accidentally includes them at the end.
    for a in authors:
        if a:
            title = re.sub(r"\s+" + re.escape(a) + r"\s*$", "", title, flags=re.I)
    title = re.sub(r"\s+", " ", title).strip(" -–—:;")
    return smart_title_case(title)


def title_ends_like_truncated_phrase(title):
    title = clean_scalar(title) or ""
    if not title:
        return False
    last = re.sub(r"[^A-Za-zÀ-ÖØ-öø-ÿ]", "", title.split()[-1]).lower()
    # Generic NL/EN linking words that usually indicate the title was cut too early.
    return last in {"van", "voor", "met", "in", "op", "aan", "om", "of", "for", "with", "in", "on", "to", "at", "by"}


def split_trailing_author_from_title(title):
    """
    Generic cleanup for front pages where OCR/PDF extraction merges title and author
    into one line. It removes a trailing person-like name from the title and returns
    (clean_title, author). No document-specific names are used.

    Important: try the shortest plausible trailing name first. This prevents title
    fragments such as "Effective Heat Stress Reduction" / "Effectieve Hittestress
    Reductie" from being swallowed into the author field.
    """
    title = clean_scalar(title)
    if not title:
        return None, None

    working = re.sub(r"\s+\d{4,12}\s*$", "", title).strip()
    tokens = working.split()
    if len(tokens) < 5:
        return clean_title_value(working), None

    # Try 2-word names first, then longer names with particles/initials.
    # This catches common cases like "Jane Smith" while still allowing
    # "Jasper van den Heuvel" or "J. van Dijk".
    max_n = min(5, len(tokens) - 3)
    for n in range(2, max_n + 1):
        candidate = " ".join(tokens[-n:])
        prefix = " ".join(tokens[:-n]).strip(" -–—:;")
        if len(prefix.split()) < 3 or len(prefix) < 12:
            continue

        previous_token = tokens[-n - 1].lower() if len(tokens) > n else ""
        # If the candidate is preceded by a linking word, it is probably still part
        # of the title, e.g. "Study of Digital Twin" should not extract "Digital Twin".
        if previous_token in SMALL_TITLE_WORDS:
            continue

        candidate_words = candidate.split()
        has_particle_or_initial = any(
            w.lower() in {"van", "de", "den", "der", "ten", "ter", "von", "da", "di", "la", "le", "du", "del"}
            or re.fullmatch(r"[A-Z]\.?", w)
            for w in candidate_words
        )
        # For 4-5 word candidates, require a particle/initial. Otherwise title
        # fragments are too easily classified as author names.
        if n >= 4 and not has_particle_or_initial:
            continue

        if looks_like_person_name(candidate):
            clean_prefix = clean_title_value(prefix)
            clean_author = smart_title_case(candidate)
            return clean_prefix, clean_author

    return clean_title_value(working), None


def extract_title_author_date_from_frontmatter(front_text):
    raw_lines = [clean_scalar(x) for x in str(front_text or "").splitlines()]
    lines = [x for x in raw_lines if x]
    front_lines = []
    for line in lines[:40]:
        low = line.lower().strip(":")
        if low in TITLE_STOP_SECTIONS or any(low.startswith(s + " ") for s in TITLE_STOP_SECTIONS):
            break
        # skip obvious page markers / file noise
        if re.fullmatch(r"\d+\s*\|\s*page", low):
            continue
        front_lines.append(line)

    date = None
    date_idx = None
    for i, line in enumerate(front_lines):
        d = extract_date_from_line(line)
        if d:
            date = d
            date_idx = i
            break

    # Usually title and author appear before the first date. If no date exists, inspect first 8 lines.
    before_date = front_lines[:date_idx] if date_idx is not None else front_lines[:8]
    before_date = [x for x in before_date if not is_identifier_line(x) and not is_org_line(x) and not extract_date_from_line(x)]

    # Author: prefer the last person-like line before a date; otherwise use explicit author-like line in early matter.
    author = None
    author_idx = None
    for i in range(len(before_date) - 1, -1, -1):
        if looks_like_person_name(before_date[i]):
            author = smart_title_case(before_date[i])
            author_idx = i
            break

    if author_idx is not None:
        title_lines = before_date[:author_idx]
    else:
        # Use the first 1-3 non-org/non-date lines as title continuation.
        title_lines = before_date[:3]

    # Remove obvious non-title lines and very short fragments.
    clean_title_lines = []
    for line in title_lines:
        if len(line) < 4:
            continue
        if looks_like_person_name(line) and len(clean_title_lines) > 0:
            continue
        clean_title_lines.append(line)

    title = None
    if clean_title_lines:
        title = ": ".join(smart_title_case(x) for x in clean_title_lines[:3])
        title = clean_title_value(title, authors=[author] if author else [], dates=[date] if date else [])

    # Generic fallback: sometimes PDF extraction merges the final title line with
    # the author name. Split a trailing person-like phrase from the title.
    split_title, split_author = split_trailing_author_from_title(title)
    if split_title:
        title = split_title
    if not author and split_author:
        author = split_author

    return {"title": title, "author": author, "date": date, "front_lines": front_lines}


def is_bad_keyword_value(value):
    value = clean_scalar(value)
    if not value:
        return True
    low = value.lower()
    if low in GENERIC_KEYWORD_STOPWORDS:
        return True
    if any(m in low for m in BAD_KEYWORD_MARKERS):
        return True
    if re.fullmatch(r"\d+(?:[.,]\d+)?", low):
        return True
    words = re.findall(r"[A-Za-zÀ-ÖØ-öø-ÿ0-9\-]+", low)
    if not words:
        return True
    if len(words) > 7:
        return True
    stop_ratio = sum(1 for w in words if w in GENERIC_KEYWORD_STOPWORDS) / max(1, len(words))
    return stop_ratio > 0.55


def clean_keywords(values, fallback_terms=None, min_items=10, max_items=15):
    out = []
    seen = set()
    for v in list(values or []) + list(fallback_terms or []):
        v = clean_scalar(v)
        if is_bad_keyword_value(v):
            continue
        key = v.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(v)
        if len(out) >= max_items:
            break
    return out


def grounded_list(values, max_items=15):
    out = []
    seen = set()
    for v in values:
        v = clean_scalar(v)
        if not v:
            continue
        key = v.lower()
        if key in seen:
            continue
        if len(v.split()) >= 2 and not evidence_contains(v):
            continue
        seen.add(key)
        out.append(v)
        if len(out) >= max_items:
            break
    return out


# Normalize schema keys and remove deprecated/unused fields.
for key, default in schema.items():
    metadata.setdefault(key, default)
metadata.pop("subtitle", None)
metadata.pop("organizations", None)

metadata["title"] = clean_scalar(metadata.get("title"))
metadata["date"] = clean_scalar(metadata.get("date"))
metadata["document_type"] = clean_scalar(metadata.get("document_type"))
metadata["language"] = clean_scalar(metadata.get("language")) or detected_language
metadata["research_or_project_topic"] = clean_scalar(metadata.get("research_or_project_topic"))
metadata["research_question_or_goal"] = clean_scalar(metadata.get("research_question_or_goal"))
metadata["short_summary"] = clean_scalar(metadata.get("short_summary"))

metadata["authors"] = ensure_list(metadata.get("authors"))
metadata["keywords"] = ensure_list(metadata.get("keywords"))
metadata["tools_technologies_or_models"] = ensure_list(metadata.get("tools_technologies_or_models"))
metadata["main_outputs_or_results"] = ensure_list(metadata.get("main_outputs_or_results"))
metadata["confidence_notes"] = ensure_list(metadata.get("confidence_notes"))

front_parse = extract_title_author_date_from_frontmatter(titlepage_excerpt)
front_title = front_parse.get("title")
front_author = front_parse.get("author")
front_date = front_parse.get("date")

# Deterministic title correction. Prefer clean front-matter title when the model title includes
# date/id/author/org noise, looks truncated, or is shorter than a clean front-matter title.
current_title = metadata.get("title")
current_authors = metadata.get("authors") or ([front_author] if front_author else [])

# Generic fix: if the model merged a trailing author name into the title, split it.
split_model_title, split_model_author = split_trailing_author_from_title(current_title)
if split_model_author:
    current_title = split_model_title
    if not any(split_model_author.lower() == str(a).lower() for a in current_authors):
        current_authors.append(split_model_author)

noise_in_title = False
if current_title:
    if any(extract_date_from_line(part) for part in [current_title]):
        noise_in_title = True
    if re.search(r"\b\d{4,12}\b", current_title):
        noise_in_title = True
    if is_org_line(current_title):
        noise_in_title = True
    if title_ends_like_truncated_phrase(current_title):
        noise_in_title = True
    for a in current_authors:
        if a and a.lower() in current_title.lower():
            noise_in_title = True

# Prefer the deterministic front-matter title if it is cleaner/more complete.
if front_title and (
    not current_title
    or noise_in_title
    or len(current_title) < len(front_title) * 0.80
    or len(current_title) > len(front_title) * 1.35
):
    metadata["title"] = front_title
else:
    metadata["title"] = clean_title_value(current_title, authors=current_authors, dates=[metadata.get("date"), front_date])

# Author validation: if the model extracted an overlong "author" that still looks like a title,
# replace it with the clean front-matter author when available.
def author_value_is_probably_title_fragment(value):
    value = clean_scalar(value)
    if not value:
        return True
    words = value.split()
    if len(words) > 4 and not any(w.lower() in {"van", "de", "den", "der", "ten", "ter", "von", "da", "di", "la", "le", "du", "del"} for w in words):
        return True
    if any(w.lower() in {"technologie", "technology", "reductie", "reduction", "effectieve", "effective", "analysis", "onderzoek", "research", "development", "ontwikkeling"} for w in words):
        return True
    return not looks_like_person_name(value)

cleaned_authors = []
for a in current_authors:
    a = smart_title_case(a)
    if not author_value_is_probably_title_fragment(a):
        cleaned_authors.append(a)
if front_author and not any(front_author.lower() == str(a).lower() for a in cleaned_authors):
    cleaned_authors.append(front_author)
metadata["authors"] = grounded_list(cleaned_authors, 10)

if not metadata["date"] and front_date:
    metadata["date"] = front_date
if not metadata["date"]:
    dc = regex_candidates.get("date_candidates", [])
    full_dates = [d for d in dc if extract_date_from_line(d)]
    metadata["date"] = full_dates[0] if full_dates else None

# Remove contact info unless present in title/front evidence.
contact = metadata.get("contact") or {}
email = clean_scalar(contact.get("email"))
phone = clean_scalar(contact.get("phone"))
name = clean_scalar(contact.get("name"))

if email and not evidence_contains(email):
    metadata["confidence_notes"].append("E-mailadres verwijderd omdat het niet in de bewijscontext stond." if str(detected_language).lower().startswith("nl") else "Removed email because it was not found in the evidence.")
    email = None
if phone and not evidence_contains(phone):
    metadata["confidence_notes"].append("Telefoonnummer verwijderd omdat het niet in de bewijscontext stond." if str(detected_language).lower().startswith("nl") else "Removed phone number because it was not found in the evidence.")
    phone = None
if name and not evidence_contains(name):
    name = None
metadata["contact"] = {"name": name, "email": email, "phone": phone}

metadata["tools_technologies_or_models"] = grounded_list(metadata["tools_technologies_or_models"], 15)
metadata["main_outputs_or_results"] = grounded_list(metadata["main_outputs_or_results"], 15)

# Use Gold top terms as fallback keywords, with generic filtering.
fallback_terms = []
for t in gold_top_terms:
    if isinstance(t, dict):
        fallback_terms.append(t.get("term"))
    else:
        fallback_terms.append(t)
metadata["keywords"] = clean_keywords(metadata.get("keywords", []), fallback_terms=fallback_terms, min_items=10, max_items=15)

# Force KMP fields and omit organizations by design.
kmp = metadata.get("suitable_kmp_fields") or {}
if not isinstance(kmp, dict):
    kmp = {}

kmp["title"] = metadata.get("title")
kmp["description"] = metadata.get("short_summary")
kmp["keywords"] = metadata.get("keywords")
kmp["contributors"] = metadata.get("authors")
kmp["date"] = metadata.get("date")
kmp["language"] = metadata.get("language")
kmp["document_type"] = metadata.get("document_type")
kmp.pop("organizations", None)
metadata["suitable_kmp_fields"] = kmp

metadata["@pipeline"] = {
    "document_id": DOCUMENT_ID,
    "processing_layer": "gold_meta",
    "processing_version": "gold_meta_local_llm_v8_generic_title_author_final_ollama",
    "created_at": datetime.now().isoformat(),
    "runtime_seconds": runtime,
    "local_execution_note": "This layer uses a local Ollama model. It can be upgraded by changing LOCAL_MODEL.",
    "model": LOCAL_MODEL,
    "source_gold_version": gold.get("@pipeline", {}).get("processing_version") or gold.get("processing_version"),
    "source_silver_version": silver.get("processing_version"),
    "source_silver_nlp_version": silver_nlp.get("processing_version"),
    "important_note": "Metadata is extracted from document evidence. Silver NLP entities are used as suggestions only. Organizations are omitted from KMP metadata by design. Title cleanup uses generic front-matter rules only."
}

meta_json_path = META_DIR / f"{DOCUMENT_ID}_gold_metadata.json"
write_json(metadata, meta_json_path)

compat_path = META_DIR / f"{DOCUMENT_ID}_meta.json"
write_json(metadata, compat_path)

# Human readable TXT
lines = []
lines.append(f"# Gold Metadata - {DOCUMENT_ID}\n")
lines.append(f"**Title:** {metadata.get('title') or '-'}")
lines.append(f"**Authors:** {', '.join(metadata.get('authors', [])) or '-'}")
lines.append(f"**Date:** {metadata.get('date') or '-'}")
lines.append(f"**Document type:** {metadata.get('document_type') or '-'}")
lines.append(f"**Language:** {metadata.get('language') or '-'}")
lines.append(f"**Project/research topic:** {metadata.get('research_or_project_topic') or '-'}")
lines.append(f"**Research question/goal:** {metadata.get('research_question_or_goal') or '-'}")

lines.append("\n## Short summary")
lines.append(metadata.get("short_summary") or "-")

lines.append("\n## Keywords")
for x in metadata.get("keywords", []):
    lines.append(f"- {x}")

lines.append("\n## Tools, technologies or models")
for x in metadata.get("tools_technologies_or_models", []):
    lines.append(f"- {x}")

lines.append("\n## Main outputs or results")
for x in metadata.get("main_outputs_or_results", []):
    lines.append(f"- {x}")

lines.append("\n## Suitable KMP fields")
for k, v in metadata.get("suitable_kmp_fields", {}).items():
    lines.append(f"- **{k}:** {v}")

lines.append("\n## Confidence")
for x in metadata.get("confidence_notes", []):
    lines.append(f"- {x}")

meta_txt_path = META_DIR / f"{DOCUMENT_ID}_gold_metadata.txt"
meta_txt_path.write_text("\n".join(lines), encoding="utf-8")

print("Front-matter parse:", front_parse)
print("Wrote:")
print(meta_json_path)
print(compat_path)
print(meta_txt_path)


Front-matter parse: {'title': 'A Deep Neural Network for: the Detection of Epileptic Seizures: in Comatose', 'author': 'Angélique Huige (80639)', 'date': None, 'front_lines': ['A Deep Neural Network for', 'the Detection of Epileptic Seizures', 'in Comatose Pediatric Patients', 'BSc Graduation Thesis', 'Angélique Huige (80639)', 'January 14th, 2024', 'Submitted in partial fulfillment of the requirements', 'for the degree of BSc Data Science', 'Supervised by:', 'Robert van den Berg, MD, PhD (Erasmus MC)', 'Bente Sinke, MSc (HZ UAS)', 'Jolène Cijsouw, MSc (HZ UAS)', 'Erasmus MC', 'Department of Neurology', 'HZ University of Applied Sciences', 'Information and Communication Technology']}
Wrote:
..\..\Data\gold_meta\doc_01_gold_metadata.json
..\..\Data\gold_meta\doc_01_meta.json
..\..\Data\gold_meta\doc_01_gold_metadata.txt
